In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import re
import random
import gc
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

# Try detect tree-sitter (optional)
try:
    from tree_sitter import Parser
    import tree_sitter_languages
    TREE_SITTER_AVAILABLE = True
except Exception:
    TREE_SITTER_AVAILABLE = False

print('TREE_SITTER_AVAILABLE =', TREE_SITTER_AVAILABLE)

# Constants
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "model.pth")
LANGUAGE_METRICS_PATH = os.path.join(OUTPUT_DIR, "language_metrics.json")
MODEL_NAME = "microsoft/unixcoder-base"

MAX_LEN = 512
BATCH_SIZE = 8
ACCUMULATION_STEPS = 2
EPOCHS = 5
PATIENCE = 3
LEARNING_RATE = 2e-5
CLASS_NAMES = ['Human-Written', 'Machine-Generated']

d:\deeplearning-assignment\binary-machine-generated-code-detection\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TREE_SITTER_AVAILABLE = True


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"Seed set to {seed}")

In [3]:
class ASTParser:
    """Wrapper to parse code to flattened AST token-types using tree-sitter (if available)."""
    def __init__(self):
        self.parsers = {}
        self.lang_map = {
            'Python': 'python', 'Java': 'java', 'C++': 'cpp', 'Go': 'go',
            'PHP': 'php', 'JavaScript': 'javascript', 'C': 'c', 'C#': 'c_sharp', 'Ruby': 'ruby'
        }

    def get_parser(self, lang_name):
        if not TREE_SITTER_AVAILABLE:
            return None
        ts_lang = self.lang_map.get(lang_name)
        if not ts_lang:
            return None
        if ts_lang in self.parsers:
            return self.parsers[ts_lang]
        try:
            parser = Parser()
            language = tree_sitter_languages.get_language(ts_lang)
            parser.set_language(language)
            self.parsers[ts_lang] = parser
            return parser
        except Exception:
            self.parsers[ts_lang] = None
            return None

    def parse_to_flattened_ast(self, code, lang_name):
        parser = self.get_parser(lang_name)
        if not parser:
            return ''
        try:
            tree = parser.parse(bytes(code, 'utf8'))
            cursor = tree.walk()
            tokens = []
            visited_children = False
            while True:
                if not visited_children:
                    if cursor.node.is_named:
                        tokens.append(cursor.node.type)
                    if cursor.goto_first_child():
                        continue
                if cursor.goto_next_sibling():
                    visited_children = False
                elif cursor.goto_parent():
                    visited_children = True
                else:
                    break
            return ' '.join(tokens)
        except Exception:
            return ''

In [4]:
# Normalization utilities (heuristic) - language-agnostic best-effort
COMMON_KEYWORDS = set([
    'def','function','fun','fn','class','if','else','elif','switch','case','default',
    'for','while','do','return','break','continue','try','except','catch','finally','throw','throws',
    'var','let','const','int','float','double','long','short','char','boolean','bool','void','public','private','protected','static','native',
    'import','from','package','include','#include','using','namespace',
    'true','false','True','False','null','None','nil',
    'new','this','super','self','async','await','lambda','yield','interface','extends','implements','abstract','final'
])
IDENT_RE = re.compile(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b')

def remove_comments(code: str) -> str:
    if not code:
        return ''
    code = re.sub(r"('{3}|\"{3})([\s\S]*?)\1", ' ', code)  # triple-quoted strings/docstrings
    code = re.sub(r'/\*[\s\S]*?\*/', ' ', code)  # C-style block
    code = re.sub(r'//.*', ' ', code)  # C++-style
    code = re.sub(r'#.*', ' ', code)  # hash comments
    return code

def mask_strings_and_literals(code: str) -> str:
    code = re.sub(r'(?s)(\"(?:\\.|[^\"\\])*\")|(\'(?:\\.|[^\'\\])*\')', ' __STR__ ', code)
    code = re.sub(r'\b0x[0-9a-fA-F]+\b', ' __NUM__ ', code)
    code = re.sub(r'\b\d+\.\d+\b', ' __NUM__ ', code)
    code = re.sub(r'\b\d+\b', ' __NUM__ ', code)
    code = re.sub(r'\b(true|false|True|False)\b', ' __BOOL__ ', code)
    return code

def normalize_operators(code: str) -> str:
    ops = {
        '&&': ' AND ', '||': ' OR ', '===': ' = ', '==': ' = ', '!==': ' != ', '!=': ' != ',
        '<=': ' <= ', '>=': ' >= ', '<': ' < ', '>': ' > ', r'\+': ' + ', '-': ' - ', r'\*': ' * ', '/': ' / '
    }
    for k, v in ops.items():
        code = re.sub(re.escape(k), v, code)
    return code

def normalize_blocks(code: str) -> str:
    code = code.replace('{', ' BLOCK_START ')
    code = code.replace('}', ' BLOCK_END ')
    code = re.sub(r':\s*(?=\n)', ' BLOCK_START ', code)  # Python-like blocks
    code = re.sub(r'\(|\)', ' PAREN ', code)
    return code

def map_identifiers(code: str, extra_keywords=None):
    keywords = set(COMMON_KEYWORDS)
    if extra_keywords:
        keywords.update(extra_keywords)
    id_map = {}
    counter = 0
    def repl(m):
        nonlocal counter
        tok = m.group(0)
        if tok in keywords:
            return ' ' + tok.upper() + ' '
        if tok not in id_map:
            counter += 1
            id_map[tok] = f'VAR_{counter}'
        return ' ' + id_map[tok] + ' '
    mapped = re.sub(IDENT_RE, repl, code)
    return mapped, id_map

In [5]:
def ast_flatten_with_tree_sitter(code: str, lang: str) -> str:
    if not TREE_SITTER_AVAILABLE or not lang:
        return ''
    lang_map = {
        'python': 'python', 'java': 'java', 'cpp': 'cpp', 'c++': 'cpp', 'c': 'c',
        'javascript': 'javascript', 'js': 'javascript', 'go': 'go', 'php': 'php', 'rust': 'rust'
    }
    ts_name = lang_map.get(lang.lower())
    if not ts_name:
        return ''
    try:
        parser = Parser()
        language = tree_sitter_languages.get_language(ts_name)
        parser.set_language(language)
        tree = parser.parse(bytes(code, 'utf8'))
        cursor = tree.walk()
        tokens = []
        visited_children = False
        while True:
            if not visited_children:
                if cursor.node.is_named:
                    tokens.append(cursor.node.type)
                if cursor.goto_first_child():
                    continue
            if cursor.goto_next_sibling():
                visited_children = False
            elif cursor.goto_parent():
                visited_children = True
            else:
                break
        return ' '.join(tokens)
    except Exception:
        return ''

def normalize_to_pseudocode(code: str, lang_hint: str = None) -> dict:
    """Return {'pseudocode':str,'identifier_map':dict,'method':'ast'|'heuristic'|'empty'}"""
    if not code:
        return {'pseudocode': '', 'identifier_map': {}, 'method': 'empty'}

    # try AST first for structural abstraction
    ast_tokens = ast_flatten_with_tree_sitter(code, lang_hint)
    if ast_tokens:
        ast_pseudo = re.sub(r'\s+', ' ', ast_tokens).strip()
        return {'pseudocode': ast_pseudo, 'identifier_map': {}, 'method': 'ast'}

    # heuristic pipeline
    code_nocom = remove_comments(code)
    code_mask = mask_strings_and_literals(code_nocom)
    code_ops = normalize_operators(code_mask)
    code_blocks = normalize_blocks(code_ops)
    pseudocode, id_map = map_identifiers(code_blocks)
    pseudocode = re.sub(r'\s+', ' ', pseudocode).strip()
    return {'pseudocode': pseudocode, 'identifier_map': id_map, 'method': 'heuristic'}

In [6]:
def clean_code_strict(code):
    if pd.isna(code) or code == '':
        return ''
    code = str(code)
    code = re.sub(r'<[^>]+>', '', code)
    lines = [line.rstrip() for line in code.split('\n') if line.strip()]
    return '\n'.join(lines)

In [7]:
class CodeDatasetNormalized(Dataset):
    """Dataset that optionally uses normalized pseudocode instead of raw code for tokenization."""
    def __init__(self, data_source, tokenizer, max_len, use_pseudocode=True, lang_weights=None):
        self.data_source = data_source
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.use_pseudocode = use_pseudocode
        self.lang_weights = lang_weights
        self.ast_parser = ASTParser()

    def __len__(self):
        return len(self.data_source)

    def __getitem__(self, index):
        item = self.data_source[index]
        raw = clean_code_strict(str(item['code']))
        label = item['label']
        language = item.get('language', '')

        if self.use_pseudocode:
            norm = normalize_to_pseudocode(raw, language)
            text_to_tokenize = norm['pseudocode']
        else:
            text_to_tokenize = raw

        if not text_to_tokenize:
            text_to_tokenize = ''

        encoding = self.tokenizer(
            text_to_tokenize,
            add_special_tokens=True,
            max_length=self.max_len,
            padding=False,
            truncation='longest_first',
            return_tensors=None
        )

        weight = 1.0
        if self.lang_weights:
            weight = self.lang_weights.get(language, 1.0)

        return {
            'input_ids': torch.tensor(encoding['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(encoding['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.float),
            'sample_weight': torch.tensor(weight, dtype=torch.float)
        }

In [8]:
def dynamic_collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    labels = torch.stack([item['labels'] for item in batch])
    weights = torch.stack([item['sample_weight'] for item in batch])
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=1)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    return {
        'input_ids': input_ids_padded,
        'attention_mask': attention_mask_padded,
        'labels': labels,
        'sample_weight': weights
    }

def compute_language_weights(raw_data):
    langs = [item['language'] for item in raw_data]
    count = Counter(langs)
    total = len(langs)
    weights = {l: total / (len(count) * freq) for l, freq in count.items()}
    return weights

In [9]:
class UniXcoderClassifier(nn.Module):
    def __init__(self, base_model):
        super(UniXcoderClassifier, self).__init__()
        self.bert = base_model
        self.drop = nn.Dropout(p=0.1)
        hidden_size = self.bert.config.hidden_size
        self.out = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        output = self.drop(pooled_output)
        return self.out(output)

In [10]:
def train_epoch(model, data_loader, loss_fn, optimizer, scaler, device):
    model = model.train()
    losses = []
    optimizer.zero_grad()
    for i, d in tqdm(enumerate(data_loader), total=len(data_loader), desc='Train', leave=False):
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        targets = d['labels'].to(device)
        weights = d['sample_weight'].to(device)
        with autocast():
            outputs = model(input_ids, attention_mask).view(-1)
            loss = (loss_fn(outputs, targets) * weights).mean() / ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        losses.append(loss.item() * ACCUMULATION_STEPS)
    return np.mean(losses) if losses else 0

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses, preds, targets = [], [], []
    with torch.no_grad():
        for d in tqdm(data_loader, desc='Eval', leave=False):
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            y = d['labels'].to(device)
            with autocast():
                out = model(input_ids, attention_mask).view(-1)
                loss = loss_fn(out, y).mean()
            losses.append(loss.item())
            preds.extend((torch.sigmoid(out) > 0.5).float().cpu().numpy())
            targets.extend(y.cpu().numpy())
    return np.mean(losses), f1_score(targets, preds, average='binary')

def get_predictions(model, data_loader, device):
    model = model.eval()
    predictions = []
    real_values = []
    with torch.no_grad():
        for d in tqdm(data_loader, desc='Predicting', leave=False):
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['labels'].to(device)
            with autocast():
                outputs = model(input_ids, attention_mask).view(-1)
                preds = (torch.sigmoid(outputs) > 0.5).float()
            predictions.extend(preds.cpu())
            real_values.extend(targets.cpu())
    return torch.stack(predictions), torch.stack(real_values)

In [11]:
def save_training_history(history, save_dir):
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'loss_history.png'))
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(history['val_f1'], label='Validation F1', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'f1_history.png'))
    plt.close()

def save_confusion_matrix_plot(y_true, y_pred, class_names, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig(os.path.join(save_dir, 'confusion_matrix.png'))
    plt.close()

In [12]:
def main(use_pseudocode=True, subsample=True, train_sample=50, val_sample=20):
    set_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Attempt to load RawCodeDataset from repo (same logic as trainer.py)
    try:
        try:
            from src.dataset import RawCodeDataset
        except Exception:
            from dataset import RawCodeDataset
    except Exception as e:
        raise ImportError('RawCodeDataset not found. Place dataset class in src/ or root, or build your own list of dicts.')

    train_raw = RawCodeDataset('train', subsample=subsample, sample_size=train_sample)
    val_raw = RawCodeDataset('validation', subsample=subsample, sample_size=val_sample)
    test_raw = RawCodeDataset('test', subsample=False)

    train_data = [{'code': x['code'], 'label': x['label'], 'language': x['language']} for x in train_raw]
    val_data = [{'code': x['code'], 'label': x['label'], 'language': x['language']} for x in val_raw]
    test_data = [{'code': x['code'], 'label': x['label'], 'language': x['language']} for x in test_raw]

    lang_weights = compute_language_weights(train_data)

    train_set = CodeDatasetNormalized(train_data, tokenizer, MAX_LEN, use_pseudocode=use_pseudocode, lang_weights=lang_weights)
    val_set = CodeDatasetNormalized(val_data, tokenizer, MAX_LEN, use_pseudocode=use_pseudocode)
    test_set = CodeDatasetNormalized(test_data, tokenizer, MAX_LEN, use_pseudocode=use_pseudocode)

    train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True, collate_fn=dynamic_collate_fn, num_workers=2)
    val_loader = DataLoader(val_set, BATCH_SIZE, collate_fn=dynamic_collate_fn, num_workers=2)
    test_loader = DataLoader(test_set, BATCH_SIZE, collate_fn=dynamic_collate_fn, num_workers=2)

    print(f'Initializing {MODEL_NAME}...')
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    model = UniXcoderClassifier(base_model).to(device)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.BCEWithLogitsLoss(reduction='none').to(device)
    scaler = GradScaler()

    best_f1 = 0
    patience_cnt = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

    for epoch in range(EPOCHS):
        print(f'Epoch {epoch+1}/{EPOCHS}')
        train_loss = train_epoch(model, train_loader, loss_fn, optimizer, scaler, device)
        val_loss, val_f1 = eval_model(model, val_loader, loss_fn, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}')
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            patience_cnt = 0
            print('>> Saved Best Model')
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print('Early stopping triggered.')
                break

    print('\nSaving Plots...')
    save_training_history(history, OUTPUT_DIR)

    print('\nEvaluating on Test Set...')
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    y_pred, y_test = get_predictions(model, test_loader, device)
    y_pred_np = y_pred.numpy()
    y_test_np = y_test.numpy()
    save_confusion_matrix_plot(y_test_np, y_pred_np, CLASS_NAMES, OUTPUT_DIR)
    print('\nOverall Classification Report:')
    print(classification_report(y_test_np, y_pred_np, target_names=CLASS_NAMES))

    # Per-language metrics
    test_languages = [x['language'] for x in test_data]
    df_results = pd.DataFrame({'true': y_test_np, 'pred': y_pred_np, 'language': test_languages})
    language_metrics = {}
    for lang in df_results['language'].unique():
        subset = df_results[df_results['language'] == lang]
        acc = accuracy_score(subset['true'], subset['pred'])
        prec, rec, f1, _ = precision_recall_fscore_support(subset['true'], subset['pred'], average='binary', zero_division=0)
        language_metrics[lang] = {'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1), 'count': int(len(subset))}
        print(f"[{lang}] F1: {f1:.4f} | Acc: {acc:.4f} | Count: {len(subset)}")

    with open(LANGUAGE_METRICS_PATH, 'w') as f:
        json.dump(language_metrics, f, indent=4)
    print(f"\nLanguage metrics saved to: {LANGUAGE_METRICS_PATH}")
    print('Pipeline Finished.')

# To run: call main(), e.g. main(use_pseudocode=True)

In [13]:
# Quick normalization sanity-check (no model/network required)
# Pastikan Anda sudah menjalankan semua sel definisi fungsi sebelumnya sebelum menjalankan ini.
sample_code = '''
def add(x, y):
    # add two numbers
    return x + y

// JS example
function greet(name) {
    console.log("Hello " + name);
}
'''
print("TREE_SITTER_AVAILABLE:", TREE_SITTER_AVAILABLE)
norm = normalize_to_pseudocode(sample_code, lang_hint='python')
print("Method used:", norm.get('method'))
print("Pseudocode preview:")
print(norm['pseudocode'][:1000])
print("\nIdentifier map (some entries):")
for i, (k, v) in enumerate(norm.get('identifier_map', {}).items()):
    print(k, "->", v)
    if i >= 20:
        break

TREE_SITTER_AVAILABLE: True
Method used: ast
Pseudocode preview:
module function_definition identifier parameters identifier identifier comment block return_statement binary_operator identifier identifier ERROR identifier expression_statement identifier expression_statement call identifier ERROR identifier argument_list identifier expression_statement set call attribute identifier identifier argument_list binary_operator string string_start string_content string_end identifier ERROR

Identifier map (some entries):


d:\deeplearning-assignment\binary-machine-generated-code-detection\venv\lib\site-packages\tree_sitter\__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


In [ ]:
# Run the full training pipeline (calls main defined in the cells)
# NOTE: main will try to import RawCodeDataset from the repo and will download the model (AutoTokenizer/AutoModel).
# If RawCodeDataset isn't available you'll get an ImportError. If you just want to test normalization, use cell_run_01 or cell_run_03.

try:
    # tweak parameters as needed; reduce sample sizes for fast smoke-test
    main(use_pseudocode=True, subsample=True, train_sample=50, val_sample=20)
except Exception as e:
    # Print full exception so you can debug why there's no output
    import traceback
    print("Error occurred while running main():")
    traceback.print_exc()

Seed set to 42
Using device: cuda


INFO:src.dataset:Loading raw train dataset from hugging face
INFO:src.dataset:Creating stratification column
INFO:src.dataset:Loaded dataset with 50 examples
INFO:src.dataset:Loading raw validation dataset from hugging face
INFO:src.dataset:Creating stratification column
INFO:src.dataset:Loaded dataset with 20 examples
INFO:src.dataset:Loading raw test dataset from hugging face
INFO:src.dataset:Loaded dataset with 1000 examples


Initializing microsoft/unixcoder-base...


d:\deeplearning-assignment\binary-machine-generated-code-detection\venv\lib\site-packages\torch\nn\modules\module.py:1341: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(
C:\Users\ilham\AppData\Local\Temp\ipykernel_18112\1464841018.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1/5


In [ ]:
# Normalize a single local file and save preview (useful if RawCodeDataset not present).
# Make sure this path points to a file you have (e.g., a local copy of src.py).
path = input("Path to code file (default 'src.py'): ").strip() or "src.py"
if not os.path.exists(path):
    print("File not found:", path)
else:
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        code = f.read()
    norm = normalize_to_pseudocode(code, lang_hint=None)
    out_preview = norm['pseudocode'][:2000]
    print("Normalization method:", norm.get('method'))
    print("Preview (first 2000 chars):\n", out_preview)
    out_path = path + ".normalized.txt"
    with open(out_path, 'w', encoding='utf-8') as g:
        g.write(norm['pseudocode'])
    print("Saved normalized pseudocode to:", out_path)

In [ ]:
#gpu check
import torch
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current GPU:", torch.cuda.current_device())
    print("GPU name:", torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Number of GPUs: 1
Current GPU: 0
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU
